# HQNN EstimatorQNN

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import cat
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn import (
    Module,
    Conv2d,
    Linear,
    Dropout2d,
    NLLLoss,
    MaxPool2d,
    Flatten,
    Sequential,
    ReLU,
    Identity
)
import torch.nn.functional as F

from sklearn.metrics import classification_report, confusion_matrix

from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.circuit.library import real_amplitudes, zz_feature_map
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit import QuantumCircuit
from qiskit_machine_learning.connectors import TorchConnector

In [2]:
transform = transforms.Compose([transforms.ToTensor()])
# transform = transforms.Compose([
#     transforms.Resize((32, 32)),
#     transforms.ToTensor()
# ])

X_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
X_test = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [3]:
n_samples_train = 400
n_samples_test = 100

X_train.data = X_train.data[:n_samples_train]
X_train.targets = X_train.targets[:n_samples_train]

X_test.data = X_test.data[:n_samples_test]
X_test.targets = X_test.targets[:n_samples_test]

In [4]:
train_size = 300
val_size = 100

train_subset, val_subset = random_split(X_train, [train_size, val_size])

In [5]:
def gaussian_noise(img, sigma=0.25): # antes 0.1
    noise = torch.randn_like(img) * sigma
    return torch.clamp(img + noise, 0, 1)

def salt_pepper(img, prob=0.15): # antes 0.05
    noisy = img.clone()

    mask = torch.rand_like(img)

    noisy[mask < prob/2] = 0
    noisy[mask > 1 - prob/2] = 1

    return noisy

def speckle(img, sigma=0.35): # antes 0.2
    noise = torch.randn_like(img) * sigma
    return torch.clamp(img + img * noise, 0, 1)


def apply_mixed_noise(img):
    r = np.random.randint(3)

    # gaussian + speckle
    if r == 0:
        img = gaussian_noise(img)
        img = speckle(img)
        label = 0

    # gaussian + salt & pepper
    elif r == 1:
        img = gaussian_noise(img)
        img = salt_pepper(img)
        label = 1

    # salt & pepper + speckle
    else:
        img = salt_pepper(img)
        img = speckle(img)
        label = 2

    return img, label


In [6]:
class NoisyMNISTDataset(Dataset):
    def __init__(self, mnist_dataset):
        self.mnist = mnist_dataset

        # Aquí preprocesamos todas las imágenes y guardamos el ruido y las etiquetas
        self.noisy_images = []
        self.labels = []

        for img, _ in self.mnist:
            noisy_img, label = apply_mixed_noise(img)
            self.noisy_images.append(noisy_img)
            self.labels.append(label)

        # Comprobación de consistencia
        if len(self.noisy_images) != len(self.labels):
            raise Exception("Incompatible arrays")


    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        return self.noisy_images[idx], self.labels[idx]
    

train_dataset = NoisyMNISTDataset(train_subset)
val_dataset = NoisyMNISTDataset(val_subset)
test_dataset = NoisyMNISTDataset(X_test)

In [7]:


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))

print(images.shape)  # [32, 1, 28, 28]
print(labels.shape)


torch.Size([32, 1, 28, 28])
torch.Size([32])


---

---

In [8]:
from dataclasses import dataclass, asdict
from typing import Optional
import copy
import random
import pandas as pd
import numpy as np
import torch


@dataclass
class ExperimentConfig:
    name: str

    # Quantum architecture
    n_qubits: int = 4
    feature_map: str = "zz"
    fm_entanglement: str = "full"
    fm_reps: int = 1

    ansatz: str = "real_amplitudes"
    ansatz_entanglement: str = "reverse_linear"
    ansatz_reps: int = 1

    # Readout
    observable_mode: str = "z_individual"

    # Training
    epochs: int = 10
    lr: float = 1e-3
    batch_size: int = 32

    # Reproducibility
    seed: int = 43


CLASS_NAMES = [
    "gaussian+speckle",
    "gaussian+saltpepper",
    "saltpepper+speckle",
]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def config_to_dict(cfg):
    return asdict(cfg)

In [9]:
from qiskit.circuit.library import (
    zz_feature_map,
    pauli_feature_map,
    real_amplitudes,
    efficient_su2,
)
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN


def build_feature_map(cfg):
    if cfg.feature_map == "zz":
        return zz_feature_map(
            cfg.n_qubits,
            entanglement=cfg.fm_entanglement,
            reps=cfg.fm_reps,
        )

    if cfg.feature_map == "pauli":
        return pauli_feature_map(
            cfg.n_qubits,
            paulis=["X", "Y", "ZZ"],
            entanglement=cfg.fm_entanglement,
            reps=cfg.fm_reps,
        )

    raise ValueError(
        f"Feature map desconocido: {cfg.feature_map}"
    )


def build_ansatz(cfg):
    if cfg.ansatz == "real_amplitudes":
        return real_amplitudes(
            cfg.n_qubits,
            entanglement=cfg.ansatz_entanglement,
            reps=cfg.ansatz_reps,
        )

    if cfg.ansatz == "efficient_su2":
        return efficient_su2(
            cfg.n_qubits,
            entanglement=cfg.ansatz_entanglement,
            reps=cfg.ansatz_reps,
        )

    raise ValueError(
        f"Ansatz desconocido: {cfg.ansatz}"
    )

In [10]:
from qiskit.quantum_info import SparsePauliOp


def pauli_operator(label):
    return SparsePauliOp.from_list([
        (label, 1.0)
    ])


def z_observable(n_qubits, qubit):
    label = ["I"] * n_qubits
    label[qubit] = "Z"
    return pauli_operator("".join(label))


def zz_observable(n_qubits, q1, q2):
    if q1 == q2:
        raise ValueError("ZZ requiere dos qubits diferentes.")

    label = ["I"] * n_qubits
    label[q1] = "Z"
    label[q2] = "Z"

    return pauli_operator("".join(label))


def _z_sum_observable(n_qubits, q1, q2):
    return (
        z_observable(n_qubits, q1)
        + z_observable(n_qubits, q2)
    )


def build_observables(cfg):
    n = cfg.n_qubits

    if cfg.observable_mode == "z_individual":
        return [
            z_observable(n, i)
            for i in range(n)
        ]

    if cfg.observable_mode == "zz":
        return [
            zz_observable(n, i, i + 1)
            for i in range(n - 1)
        ]

    if cfg.observable_mode == "z_sum":
        return [
            _z_sum_observable(n, i, i + 1)
            for i in range(n - 1)
        ]

    if cfg.observable_mode == "correlations":
        observables = []

        # Z individuales
        for i in range(n):
            observables.append(
                z_observable(n, i)
            )

        # Correlaciones ZZ
        for i in range(n - 1):
            observables.append(
                zz_observable(n, i, i + 1)
            )

        return observables

    raise ValueError(
        f"Observable mode desconocido: {cfg.observable_mode}"
    )

In [11]:
def build_qnn(cfg):

    estimator = Estimator()

    feature_map = build_feature_map(cfg)
    ansatz = build_ansatz(cfg)
    observables = build_observables(cfg)

    qc = QuantumCircuit(cfg.n_qubits)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        input_gradients=True,
        estimator=estimator,
        observables=observables,
    )

    return {
        "qnn": qnn,
        "circuit": qc,
        "feature_map": feature_map,
        "ansatz": ansatz,
        "observables": observables,
    }

In [12]:
class HQCNN(nn.Module):

    def __init__(self, qnn, n_qubits, n_observables):
        super().__init__()

        self.conv1 = nn.Conv2d(
            1, 16, kernel_size=5, padding=2
        )

        self.conv2 = nn.Conv2d(
            16, 32, kernel_size=5, padding=2
        )

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(
            32 * 7 * 7,
            128
        )

        self.fc2 = nn.Linear(
            128,
            n_qubits
        )

        self.qnn = TorchConnector(qnn)

        self.classifier = nn.Linear(
            n_observables,
            3
        )

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.shape[0], -1)

        x = F.relu(self.fc1(x))

        x = torch.tanh(self.fc2(x)) * np.pi

        x = self.qnn(x)

        x = self.classifier(x)

        return x

In [13]:
EXPERIMENTS_A = [

    ExperimentConfig(
        name="A1",
        n_qubits=4,
        feature_map="zz",
        ansatz="real_amplitudes",
        observable_mode="z_individual",
    ),

    ExperimentConfig(
        name="A2",
        n_qubits=4,
        feature_map="zz",
        ansatz="efficient_su2",
        observable_mode="z_individual",
    ),

    ExperimentConfig(
        name="A3",
        n_qubits=4,
        feature_map="pauli",
        ansatz="real_amplitudes",
        observable_mode="z_individual",
    ),

    ExperimentConfig(
        name="A4",
        n_qubits=4,
        feature_map="pauli",
        ansatz="efficient_su2",
        observable_mode="z_individual",
    ),
]

In [14]:
def train_model(
    model,
    train_loader,
    val_loader,
    cfg,
    device,
):

    model.to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=cfg.lr
    )

    criterion = nn.CrossEntropyLoss()

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(cfg.epochs):

        model.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for data, target in train_loader:

            data = data.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            output = model(data)

            loss = criterion(
                output,
                target
            )

            loss.backward()
            optimizer.step()

            total_loss += (
                loss.item() * data.size(0)
            )

            predicted = output.argmax(1)

            correct += (
                predicted == target
            ).sum().item()

            total += target.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        model.eval()

        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():

            for data, target in val_loader:

                data = data.to(device)
                target = target.to(device)

                output = model(data)

                loss = criterion(
                    output,
                    target
                )

                val_loss += (
                    loss.item() * data.size(0)
                )

                predicted = output.argmax(1)

                correct += (
                    predicted == target
                ).sum().item()

                total += target.size(0)

        val_loss /= total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"{cfg.name} | "
            f"Epoch {epoch+1}/{cfg.epochs} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

    history["best_val_acc"] = max(
        history["val_acc"]
    )

    history["best_epoch"] = (
        int(np.argmax(history["val_acc"])) + 1
    )

    return history

In [15]:
def evaluate_model(model, test_loader, device):

    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():

        for data, target in test_loader:

            data = data.to(device)
            target = target.to(device)

            output = model(data)

            preds = output.argmax(1)

            all_preds.append(
                preds.cpu()
            )

            all_targets.append(
                target.cpu()
            )

    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)

    accuracy = (
        preds == targets
    ).float().mean().item()

    cm = confusion_matrix(
        targets,
        preds
    )

    report = classification_report(
        targets,
        preds,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "report": report,
    }

In [16]:
def run_experiment(
    cfg,
    train_loader,
    val_loader,
    test_loader,
    device,
):

    set_seed(cfg.seed)

    bundle = build_qnn(cfg)

    model = HQCNN(
        qnn=bundle["qnn"],
        n_qubits=cfg.n_qubits,
        n_observables=len(bundle["observables"]),
    )

    history = train_model(
        model,
        train_loader,
        val_loader,
        cfg,
        device,
    )

    result = evaluate_model(
        model,
        test_loader,
        device,
    )

    return {
        "config": cfg,
        "bundle": bundle,
        "model": model,
        "history": history,
        "result": result,
    }

In [18]:
import torch
print(torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


True


In [19]:
results_A = []

for cfg in EXPERIMENTS_A:

    print("=" * 70)
    print(f"RUNNING {cfg.name}")
    print("=" * 70)

    result = run_experiment(
        cfg,
        train_loader,
        val_loader,
        test_loader,
        device,
    )

    results_A.append(result)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING A1
A1 | Epoch 1/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A1 | Epoch 2/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A1 | Epoch 3/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A1 | Epoch 4/10 | Train Acc: 0.3367 | Val Acc: 0.3400
A1 | Epoch 5/10 | Train Acc: 0.3367 | Val Acc: 0.2900
A1 | Epoch 6/10 | Train Acc: 0.4600 | Val Acc: 0.3500
A1 | Epoch 7/10 | Train Acc: 0.5400 | Val Acc: 0.5000
A1 | Epoch 8/10 | Train Acc: 0.8100 | Val Acc: 0.8100
A1 | Epoch 9/10 | Train Acc: 0.8800 | Val Acc: 0.9900
A1 | Epoch 10/10 | Train Acc: 0.9600 | Val Acc: 0.6900


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING A2
A2 | Epoch 1/10 | Train Acc: 0.3700 | Val Acc: 0.3300
A2 | Epoch 2/10 | Train Acc: 0.4000 | Val Acc: 0.4500
A2 | Epoch 3/10 | Train Acc: 0.4033 | Val Acc: 0.3500
A2 | Epoch 4/10 | Train Acc: 0.4433 | Val Acc: 0.4300
A2 | Epoch 5/10 | Train Acc: 0.5200 | Val Acc: 0.5000
A2 | Epoch 6/10 | Train Acc: 0.5467 | Val Acc: 0.4400
A2 | Epoch 7/10 | Train Acc: 0.5667 | Val Acc: 0.3600
A2 | Epoch 8/10 | Train Acc: 0.3533 | Val Acc: 0.3300
A2 | Epoch 9/10 | Train Acc: 0.3500 | Val Acc: 0.3300
A2 | Epoch 10/10 | Train Acc: 0.4567 | Val Acc: 0.5700


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING A3
A3 | Epoch 1/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A3 | Epoch 2/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A3 | Epoch 3/10 | Train Acc: 0.3300 | Val Acc: 0.2900
A3 | Epoch 4/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A3 | Epoch 5/10 | Train Acc: 0.3267 | Val Acc: 0.3000
A3 | Epoch 6/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A3 | Epoch 7/10 | Train Acc: 0.3267 | Val Acc: 0.2800
A3 | Epoch 8/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A3 | Epoch 9/10 | Train Acc: 0.3267 | Val Acc: 0.2900
A3 | Epoch 10/10 | Train Acc: 0.3333 | Val Acc: 0.2900


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING A4
A4 | Epoch 1/10 | Train Acc: 0.3633 | Val Acc: 0.3300
A4 | Epoch 2/10 | Train Acc: 0.3433 | Val Acc: 0.3300
A4 | Epoch 3/10 | Train Acc: 0.3467 | Val Acc: 0.3300
A4 | Epoch 4/10 | Train Acc: 0.3500 | Val Acc: 0.3300
A4 | Epoch 5/10 | Train Acc: 0.3500 | Val Acc: 0.3300
A4 | Epoch 6/10 | Train Acc: 0.3500 | Val Acc: 0.3300
A4 | Epoch 7/10 | Train Acc: 0.3633 | Val Acc: 0.5400
A4 | Epoch 8/10 | Train Acc: 0.6267 | Val Acc: 0.6600
A4 | Epoch 9/10 | Train Acc: 0.9167 | Val Acc: 0.9600
A4 | Epoch 10/10 | Train Acc: 0.9800 | Val Acc: 0.9600


In [20]:
def results_to_dataframe(results):

    rows = []

    for r in results:

        cfg = r["config"]
        result = r["result"]
        history = r["history"]

        rows.append({
            "experiment": cfg.name,
            "qubits": cfg.n_qubits,
            "feature_map": cfg.feature_map,
            "ansatz": cfg.ansatz,
            "observable": cfg.observable_mode,
            "best_epoch": history["best_epoch"],
            "best_val_acc": history["best_val_acc"],
            "test_accuracy": result["accuracy"],
        })

    return pd.DataFrame(rows)


df_A = results_to_dataframe(results_A)

display(
    df_A.sort_values(
        "test_accuracy",
        ascending=False
    )
)

,experiment,qubits,feature_map,ansatz,observable,best_epoch,best_val_acc,test_accuracy
3,A4,4,pauli,efficient_su2,z_individual,9,0.96,0.98
0,A1,4,zz,real_amplitudes,z_individual,9,0.99,0.73
1,A2,4,zz,efficient_su2,z_individual,10,0.57,0.52
2,A3,4,pauli,real_amplitudes,z_individual,5,0.30,0.36


In [21]:
best_A = max(
    results_A,
    key=lambda r: r["result"]["accuracy"]
)

best_A_cfg = best_A["config"]

print("Mejor experimento de Fase A:")
print(best_A_cfg.name)

print(
    f"Accuracy: "
    f"{best_A['result']['accuracy']:.4f}"
)

Mejor experimento de Fase A:
A4
Accuracy: 0.9800


In [22]:
EXPERIMENTS_B = []

for name, observable in [
    ("B1", "z_individual"),
    ("B2", "zz"),
    ("B3", "z_sum"),
    ("B4", "correlations"),
]:

    cfg = copy.deepcopy(best_A_cfg)

    cfg.name = name
    cfg.observable_mode = observable

    EXPERIMENTS_B.append(cfg)

In [ ]:
results_B = []

for cfg in EXPERIMENTS_B:

    print("=" * 70)
    print(f"RUNNING {cfg.name}")
    print("=" * 70)

    result = run_experiment(
        cfg,
        train_loader,
        val_loader,
        test_loader,
        device,
    )

    results_B.append(result)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING B1
B1 | Epoch 1/10 | Train Acc: 0.3000 | Val Acc: 0.3700
B1 | Epoch 2/10 | Train Acc: 0.3600 | Val Acc: 0.2600
B1 | Epoch 3/10 | Train Acc: 0.3567 | Val Acc: 0.3700
B1 | Epoch 4/10 | Train Acc: 0.4100 | Val Acc: 0.3300
B1 | Epoch 5/10 | Train Acc: 0.3500 | Val Acc: 0.3300
B1 | Epoch 6/10 | Train Acc: 0.3500 | Val Acc: 0.3300
B1 | Epoch 7/10 | Train Acc: 0.3500 | Val Acc: 0.3300
B1 | Epoch 8/10 | Train Acc: 0.3500 | Val Acc: 0.3300
B1 | Epoch 9/10 | Train Acc: 0.3500 | Val Acc: 0.3300
B1 | Epoch 10/10 | Train Acc: 0.3500 | Val Acc: 0.3300


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING B2
B2 | Epoch 1/10 | Train Acc: 0.3100 | Val Acc: 0.2900
B2 | Epoch 2/10 | Train Acc: 0.3600 | Val Acc: 0.2900
B2 | Epoch 3/10 | Train Acc: 0.3133 | Val Acc: 0.2800
B2 | Epoch 4/10 | Train Acc: 0.3267 | Val Acc: 0.3100
B2 | Epoch 5/10 | Train Acc: 0.3400 | Val Acc: 0.2300
B2 | Epoch 6/10 | Train Acc: 0.2967 | Val Acc: 0.3800
B2 | Epoch 7/10 | Train Acc: 0.3400 | Val Acc: 0.3800
B2 | Epoch 8/10 | Train Acc: 0.3667 | Val Acc: 0.2900
B2 | Epoch 9/10 | Train Acc: 0.3733 | Val Acc: 0.3700
B2 | Epoch 10/10 | Train Acc: 0.4033 | Val Acc: 0.5800


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING B3
B3 | Epoch 1/10 | Train Acc: 0.3167 | Val Acc: 0.5100
B3 | Epoch 2/10 | Train Acc: 0.3500 | Val Acc: 0.3900
B3 | Epoch 3/10 | Train Acc: 0.4833 | Val Acc: 0.6300
B3 | Epoch 4/10 | Train Acc: 0.6000 | Val Acc: 0.6700
B3 | Epoch 5/10 | Train Acc: 0.6433 | Val Acc: 0.6700
B3 | Epoch 6/10 | Train Acc: 0.6500 | Val Acc: 0.6700
B3 | Epoch 7/10 | Train Acc: 0.6500 | Val Acc: 0.6700
B3 | Epoch 8/10 | Train Acc: 0.6500 | Val Acc: 0.6700
B3 | Epoch 9/10 | Train Acc: 0.6500 | Val Acc: 0.6700
B3 | Epoch 10/10 | Train Acc: 0.6500 | Val Acc: 0.6700


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING B4
B4 | Epoch 1/10 | Train Acc: 0.3700 | Val Acc: 0.3300
B4 | Epoch 2/10 | Train Acc: 0.3733 | Val Acc: 0.4500
B4 | Epoch 3/10 | Train Acc: 0.3167 | Val Acc: 0.4600
B4 | Epoch 4/10 | Train Acc: 0.4667 | Val Acc: 0.4800
B4 | Epoch 5/10 | Train Acc: 0.4000 | Val Acc: 0.4500
B4 | Epoch 6/10 | Train Acc: 0.6100 | Val Acc: 0.5000
B4 | Epoch 7/10 | Train Acc: 0.6033 | Val Acc: 0.6600
B4 | Epoch 8/10 | Train Acc: 0.7067 | Val Acc: 0.7700
B4 | Epoch 9/10 | Train Acc: 0.8467 | Val Acc: 0.8600
B4 | Epoch 10/10 | Train Acc: 0.8300 | Val Acc: 0.7000


In [24]:
df_B = results_to_dataframe(results_B)

display(
    df_B.sort_values(
        "test_accuracy",
        ascending=False
    )
)

,experiment,qubits,feature_map,ansatz,observable,best_epoch,best_val_acc,test_accuracy
3,B4,4,pauli,efficient_su2,correlations,9,0.86,0.79
2,B3,4,pauli,efficient_su2,z_sum,4,0.67,0.68
1,B2,4,pauli,efficient_su2,zz,10,0.58,0.56
0,B1,4,pauli,efficient_su2,z_individual,1,0.37,0.32


In [25]:
best_B = max(
    results_B,
    key=lambda r: r["result"]["accuracy"]
)

best_B_cfg = best_B["config"]

print("Mejor experimento de Fase B:")
print(best_B_cfg.name)

print(
    f"Accuracy: "
    f"{best_B['result']['accuracy']:.4f}"
)

print(
    f"Observable: "
    f"{best_B_cfg.observable_mode}"
)

Mejor experimento de Fase B:
B4
Accuracy: 0.7900
Observable: correlations


In [ ]:
EXPERIMENTS_C = []

for name, n_qubits in [
    (" C1", 4),
    ("C2", 6),
]:

    cfg = copy.deepcopy(best_B_cfg)

    cfg.name = name
    cfg.n_qubits = n_qubits

    EXPERIMENTS_C.append(cfg)

In [27]:
results_C = []

for cfg in EXPERIMENTS_C:

    print("=" * 70)
    print(f"RUNNING {cfg.name}")
    print("=" * 70)

    result = run_experiment(
        cfg,
        train_loader,
        val_loader,
        test_loader,
        device,
    )

    results_C.append(result)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING C1
C1 | Epoch 1/10 | Train Acc: 0.3500 | Val Acc: 0.3700
C1 | Epoch 2/10 | Train Acc: 0.3600 | Val Acc: 0.3300
C1 | Epoch 3/10 | Train Acc: 0.3500 | Val Acc: 0.3200
C1 | Epoch 4/10 | Train Acc: 0.4100 | Val Acc: 0.3300
C1 | Epoch 5/10 | Train Acc: 0.3567 | Val Acc: 0.1300
C1 | Epoch 6/10 | Train Acc: 0.3233 | Val Acc: 0.3800
C1 | Epoch 7/10 | Train Acc: 0.5533 | Val Acc: 0.5800
C1 | Epoch 8/10 | Train Acc: 0.6567 | Val Acc: 0.8100
C1 | Epoch 9/10 | Train Acc: 0.7033 | Val Acc: 0.6800
C1 | Epoch 10/10 | Train Acc: 0.7767 | Val Acc: 0.6500


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


RUNNING C2
C2 | Epoch 1/10 | Train Acc: 0.2833 | Val Acc: 0.3500
C2 | Epoch 2/10 | Train Acc: 0.3467 | Val Acc: 0.3600
C2 | Epoch 3/10 | Train Acc: 0.3667 | Val Acc: 0.4300
C2 | Epoch 4/10 | Train Acc: 0.5300 | Val Acc: 0.7200
C2 | Epoch 5/10 | Train Acc: 0.7167 | Val Acc: 0.7800
C2 | Epoch 6/10 | Train Acc: 0.7733 | Val Acc: 0.8600
C2 | Epoch 7/10 | Train Acc: 0.8267 | Val Acc: 0.8500
C2 | Epoch 8/10 | Train Acc: 0.8033 | Val Acc: 0.7400
C2 | Epoch 9/10 | Train Acc: 0.7733 | Val Acc: 0.8200
C2 | Epoch 10/10 | Train Acc: 0.8467 | Val Acc: 0.8300


In [28]:
df_all = pd.concat(
    [
        results_to_dataframe(results_A),
        results_to_dataframe(results_B),
        results_to_dataframe(results_C),
    ],
    ignore_index=True,
)

display(
    df_all.sort_values(
        "test_accuracy",
        ascending=False
    )
)

,experiment,qubits,feature_map,ansatz,observable,best_epoch,best_val_acc,test_accuracy
3,A4,4,pauli,efficient_su2,z_individual,9,0.96,0.98
7,B4,4,pauli,efficient_su2,correlations,9,0.86,0.79
0,A1,4,zz,real_amplitudes,z_individual,9,0.99,0.73
9,C2,6,pauli,efficient_su2,correlations,6,0.86,0.72
6,B3,4,pauli,efficient_su2,z_sum,4,0.67,0.68
8,C1,4,pauli,efficient_su2,correlations,8,0.81,0.67
5,B2,4,pauli,efficient_su2,zz,10,0.58,0.56
1,A2,4,zz,efficient_su2,z_individual,10,0.57,0.52
2,A3,4,pauli,real_amplitudes,z_individual,5,0.30,0.36
4,B1,4,pauli,efficient_su2,z_individual,1,0.37,0.32


In [29]:
def make_summary(results, phase):

    rows = []

    for r in results:

        cfg = r["config"]
        result = r["result"]
        history = r["history"]

        report = result["report"]

        rows.append({
            "phase": phase,
            "experiment": cfg.name,
            "qubits": cfg.n_qubits,
            "feature_map": cfg.feature_map,
            "ansatz": cfg.ansatz,
            "observable": cfg.observable_mode,

            "best_epoch":
                history["best_epoch"],

            "best_val_acc":
                history["best_val_acc"],

            "test_accuracy":
                result["accuracy"],

            "macro_f1":
                report["macro avg"]["f1-score"],

            "weighted_f1":
                report["weighted avg"]["f1-score"],
        })

    return pd.DataFrame(rows)

In [30]:
summary_A = make_summary(results_A, "A")
summary_B = make_summary(results_B, "B")
summary_C = make_summary(results_C, "C")

In [31]:
summary_A 

,phase,experiment,qubits,feature_map,ansatz,observable,best_epoch,best_val_acc,test_accuracy,macro_f1,weighted_f1
0,A,A1,4,zz,real_amplitudes,z_individual,9,0.99,0.73,0.673868,0.685832
1,A,A2,4,zz,efficient_su2,z_individual,10,0.57,0.52,0.480267,0.489292
2,A,A3,4,pauli,real_amplitudes,z_individual,5,0.30,0.36,0.176471,0.190588
3,A,A4,4,pauli,efficient_su2,z_individual,9,0.96,0.98,0.980375,0.980017


In [32]:
summary_B

,phase,experiment,qubits,feature_map,ansatz,observable,best_epoch,best_val_acc,test_accuracy,macro_f1,weighted_f1
0,B,B1,4,pauli,efficient_su2,z_individual,1,0.37,0.32,0.161616,0.155152
1,B,B2,4,pauli,efficient_su2,zz,10,0.58,0.56,0.459551,0.441169
2,B,B3,4,pauli,efficient_su2,z_sum,4,0.67,0.68,0.564103,0.569231
3,B,B4,4,pauli,efficient_su2,correlations,9,0.86,0.79,0.785729,0.779563


In [33]:
summary_C

,phase,experiment,qubits,feature_map,ansatz,observable,best_epoch,best_val_acc,test_accuracy,macro_f1,weighted_f1
0,C,C1,4,pauli,efficient_su2,correlations,8,0.81,0.67,0.65093,0.639010
1,C,C2,6,pauli,efficient_su2,correlations,6,0.86,0.72,0.68643,0.673518
